# A. 時間序列基礎 + STL 分解
對應敘事文件: narrative/01_basics_stl.md
主軸：**訊號常藏在「殘差／距平」裡，而不是原始序列。** 這個觀念會用三種尺度反覆出現。

註：圖上文字一律用英文，以免 Colab 預設字型缺中文而變方塊；中文說明放在 .md 與註解。

In [ ]:
# A0. 載入套件與共用工具
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

import ts_utils as ts

## A1. 先用「模擬訊號」建立直覺
自己造一條 = 趨勢 + 季節 + 噪音 的序列，因為**知道正確答案**，
才能檢查待會 STL 拆得對不對。

In [ ]:
sim = ts.make_synthetic_series(n=120, slope=0.04, season_amp=2.0, noise=0.5)

plt.figure(figsize=(12, 3))
plt.plot(sim.index, sim["y"], label="Observed  y = trend + season + noise", color="steelblue")
plt.plot(sim.index, sim["signal"], label="True signal (trend + season)", color="crimson", lw=2)
plt.title("Synthetic time series")
plt.legend()
plt.tight_layout()
plt.show()

## A2. STL 分解：把序列拆成 趨勢 / 季節 / 殘差
STL = Seasonal-Trend decomposition using LOESS。`period=12` 表示一年 12 個月。
`robust=True` 會降低離群值 (outlier) 的權重，讓擬合不易被極端值帶歪。

In [ ]:
stl_sim = STL(sim["y"], period=12, robust=True).fit()
fig = stl_sim.plot()
fig.set_size_inches(12, 8)
fig.suptitle("STL decomposition of the synthetic series", y=1.00)
plt.tight_layout()
plt.show()

## A3. 驗證：STL 拆出來的趨勢/季節，是否接近我們設定的真值？
把 STL 的 trend、seasonal 疊回造資料時的 ground truth。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sim.index, sim["trend"], label="True trend", color="crimson", lw=2)
axes[0].plot(sim.index, stl_sim.trend, label="STL-estimated trend", color="black", ls="--")
axes[0].set_title("Trend")
axes[0].legend(loc="upper left")

axes[1].plot(sim.index, sim["season"], label="True seasonal", color="crimson", lw=2)
axes[1].plot(sim.index, stl_sim.seasonal, label="STL-estimated seasonal", color="black", ls="--")
axes[1].set_title("Seasonal")
axes[1].legend(loc="upper left")
plt.tight_layout()
plt.show()

## A4. 真實資料：原始海溫 vs 距平（合併看）
換上真的海洋資料：**Niño 3.4 區域月平均海溫**（赤道中太平洋，判斷聖嬰/反聖嬰的關鍵區）。

- 上圖：原始海溫（實線）疊上「每個月該有的長期平均」(climatology，虛線)。
  有趣的是——赤道附近**季節變化其實很小**（虛線幾乎是平的，全年只差約 1.3°C），
  但原始值卻上下大幅擺盪 (約 5°C)：那些大擺盪**不是季節，而是真正的異常 (ENSO)**。
- 下圖：距平 = 原始 − 月氣候平均。把基準拉到 0 之後，就能用**一條統一門檻**
  (±0.5°C，即 ONI) 來定義聖嬰/反聖嬰——這正是下一段 (B) 的基礎；原始海溫沒辦法這樣設門檻。

In [ ]:
sst = ts.load_nino34_sst()
clim = ts.monthly_climatology(sst, base=("1982", "2011"))   # 12 個月長期平均
baseline = sst.index.month.map(clim)                        # 把月平均「鋪」回時間軸
anom = ts.to_anomaly(sst, clim)                             # 距平
print("季節 (climatology) 全年振幅僅 %.2f°C，原始值跨度卻有 %.2f°C"
      % (clim.max() - clim.min(), sst.max() - sst.min()))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sst.index, sst, color="darkorange", lw=1, label="Raw SST")
axes[0].plot(sst.index, baseline, color="gray", ls="--", lw=1.2,
             label="Monthly climatology (expected)")
axes[0].set_title("Niño 3.4 raw SST vs monthly climatology  (season is weak near the equator)")
axes[0].set_ylabel("SST (°C)")
axes[0].legend(loc="upper left")

axes[1].axhline(0, color="gray", lw=0.6)
axes[1].fill_between(anom.index, anom, 0, where=anom >= 0, color="tab:red", alpha=0.5)
axes[1].fill_between(anom.index, anom, 0, where=anom < 0, color="tab:blue", alpha=0.5)
axes[1].axhline(0.5, color="red", ls=":", lw=0.8)
axes[1].axhline(-0.5, color="blue", ls=":", lw=0.8)
axes[1].set_title("Anomaly = raw − monthly climatology   (±0.5°C dotted = ENSO threshold, next section)")
axes[1].set_ylabel("SST anomaly (°C)")
plt.tight_layout()
plt.show()

## A5. STL 也能做到——而且自動處理趨勢
直接把「原始海溫」丟給 STL：seasonal 對應手算的氣候季節，
residual 則類似「距平再去掉長期趨勢」。比較兩種做法殊途同歸。

一個重要旋鈕：`trend=` 是趨勢的平滑視窗 (月)。預設視窗較短，趨勢太有彈性，
會把我們想看的 ENSO 年際訊號也「吃進趨勢」。把它加大 (這裡 181≈15 年)，
趨勢變硬、只保留長期／多十年的緩變 (不一定單調)，ENSO 就留在殘差裡——殘差因此更貼近手算距平。

In [ ]:
stl_sst = STL(sst, period=12, trend=181, robust=True).fit()
fig = stl_sst.plot()
fig.set_size_inches(12, 8)
fig.suptitle("STL decomposition of Niño 3.4 raw SST", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# 把「手算距平」與「STL 殘差」疊起來看：兩者形狀相近 (STL 又多扣掉了趨勢)
plt.figure(figsize=(12, 3))
plt.axhline(0, color="gray", lw=0.6)
plt.plot(anom.index, anom, label="Manual anomaly (raw − monthly mean)", color="seagreen", alpha=0.8)
plt.plot(sst.index, stl_sst.resid, label="STL residual (trend also removed)", color="black", alpha=0.7)
plt.title("Anomaly vs STL residual: the signal lives in the residual")
plt.ylabel("°C")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

# B. 把序列變成「事件」，再談相關（與相關 ≠ 因果）
對應敘事文件: narrative/02_enso_events.md
時間預估：約 25 分鐘

承接 A 段主軸：距平讓 ENSO 訊號浮現。這一段把連續的距平序列**抽象成離散事件**
(聖嬰/反聖嬰)，再用事件去談相關——並親手示範「定義方式會翻轉統計結論」。

註：圖上文字一律英文 (避免 Colab 缺中文字型)；中文說明在 .md 與註解。

In [ ]:
# B0. 套件與工具
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import ts_utils as ts

## B1. 先看 Niño 3.4 在哪裡：太平洋 SST 距平地圖
用 cartopy 畫一張跨換日線的赤道太平洋距平圖（2015-12，超強聖嬰）。
技巧：經度轉 0–360 後資料是連續的（不會在 180° 斷開），
投影用 `PlateCarree(central_longitude=180)` 把太平洋擺中間，資料用 `transform=PlateCarree()`。

In [ ]:
def plot_enso_map(date="2015-12", note="strong El Niño"):
    df = ts.load_enso_map(date)            # 線上優先、失敗用快取
    df["lon360"] = df["lon"] % 360         # 轉 0–360：跨太平洋變連續
    grid = df.pivot_table(index="lat", columns="lon360", values="sst_anomaly")
    LON, LAT = np.meshgrid(grid.columns.values, grid.index.values)

    proj = ccrs.PlateCarree(central_longitude=180)
    data_crs = ccrs.PlateCarree()
    fig = plt.figure(figsize=(12, 4.5))
    ax = plt.axes(projection=proj)
    ax.set_extent([135, 300, -25, 25], crs=data_crs)

    im = ax.contourf(LON, LAT, grid.values, levels=np.linspace(-3, 3, 21),
                     cmap="RdYlBu_r", extend="both", transform=data_crs)
    ax.add_feature(cfeature.LAND.with_scale("110m"), facecolor="lightgray", zorder=3)
    ax.coastlines("110m", lw=0.6, zorder=4)

    # Niño 區域框 (經度用 0–360)
    boxes = {"Niño 4": (160, 210, -5, 5), "Niño 3.4": (190, 240, -5, 5),
             "Niño 3": (210, 270, -5, 5), "Niño 1+2": (270, 280, -10, 0)}
    for name, (a, b, c, d) in boxes.items():
        lw = 2.0 if name == "Niño 3.4" else 1.0
        ax.plot([a, b, b, a, a], [c, c, d, d, c], transform=data_crs, lw=lw, label=name)

    gl = ax.gridlines(draw_labels=True, lw=0.3, ls="--")
    gl.top_labels = gl.right_labels = False
    plt.colorbar(im, ax=ax, orientation="vertical", shrink=0.8, pad=0.02,
                 label="SST anomaly (°C)")
    ax.legend(loc="upper left", fontsize=8, ncol=4)
    ax.set_title(f"Pacific SST anomaly — {date}  ({note})")
    # 注意：cartopy 的 gridline labels 與 plt.tight_layout() 會衝突，這裡不要用 tight_layout
    plt.show()


# 想換成反聖嬰範例可改成: plot_enso_map("2007-12", note="La Niña")
plot_enso_map("2015-12", note="strong El Niño")

## B2. 連續序列 → 指數 → 事件
三步驟把「距平」變成大家講的「聖嬰年/反聖嬰年」：
1. **ONI** = Niño 3.4 距平的 3 個月移動平均（NOAA 官方定義）。
2. **門檻**：ONI ≥ +0.5°C 偏暖、≤ −0.5°C 偏冷。
3. **事件**：要連續 ≥ 5 個月超過門檻，才算一次聖嬰/反聖嬰事件（濾掉短暫雜訊）。

In [ ]:
ssta = ts.load_noaa_nino34("1950", "2025")   # NOAA Niño3.4 月距平 (線上優先/快取)
ssta["oni"] = ssta["ssta"].rolling(3, center=True).mean()


def classify_enso_events(df, min_months=5, threshold=0.5):
    """把 ONI 序列切成 聖嬰/反聖嬰 事件 (連續 ≥ min_months 個月超過門檻)。"""
    def phase_of(x):
        if x >= threshold:
            return "El Niño"
        if x <= -threshold:
            return "La Niña"
        return "Neutral"

    d = df.copy()
    d["phase"] = d["oni"].apply(phase_of)
    d["run"] = (d["phase"] != d["phase"].shift()).cumsum()   # 連續同相位編號
    events = []
    for _, g in d.groupby("run"):
        ph = g["phase"].iloc[0]
        if ph in ("El Niño", "La Niña") and len(g) >= min_months:
            peak = g["oni"].max() if ph == "El Niño" else g["oni"].min()
            events.append({"start": g.index[0], "end": g.index[-1],
                           "phase": ph, "peak_oni": peak})
    return pd.DataFrame(events)


events = classify_enso_events(ssta)
print(events.tail(6).to_string(index=False))

## B3. 事件長條圖：把整段歷史的聖嬰/反聖嬰標出來

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 1]})
# 上：ONI 曲線 + 門檻
axes[0].plot(ssta.index, ssta["oni"], color="cornflowerblue", lw=1)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].axhline(0.5, color="red", ls=":", lw=0.8)
axes[0].axhline(-0.5, color="blue", ls=":", lw=0.8)
axes[0].set_ylabel("ONI (°C)")
axes[0].set_title("Oceanic Niño Index (3-month running mean of Niño 3.4 anomaly)")

# 下：把事件期間塗成長條 (紅=聖嬰, 藍=反聖嬰)
for _, ev in events.iterrows():
    color = "tab:red" if ev["phase"] == "El Niño" else "tab:blue"
    vals = ssta.loc[ev["start"]:ev["end"], "oni"]   # 直接用真實索引切片 (day=15)
    axes[1].bar(vals.index, vals.values, width=25, color=color, alpha=0.7)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_ylabel("ENSO events\n(red=El Niño, blue=La Niña)")
axes[1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## B4. 用事件談相關——以及一個刻意的陷阱
經典問題：聖嬰會不會影響遠方的疾病？這裡用一篇研究的坦尚尼亞**霍亂**年資料
(已去趨勢) 對上 ENSO 事件。

關鍵教學點：**怎麼把「跨年的 DJF (12–1–2月) 事件」指派給哪一年，是人為決定。**
我們刻意做兩種合理的指派，看看結論會不會一樣。

In [ ]:
cholera = ts.load_cholera()


def annual_enso_phase(ssta_df, years, assign_to_next_year):
    """
    把每年冬季 DJF 的 ONI 指派成該年的 ENSO 相位。
    assign_to_next_year=False: DJF(去年12, 今年1, 今年2) 當作「今年」。
    assign_to_next_year=True : 同一個 DJF 改算給「隔年」(只差一個年份的人為選擇)。
    """
    rows = []
    for yr in years:
        base = yr - 1 if assign_to_next_year else yr
        djf_months = [pd.Timestamp(base, 12, 15),
                      pd.Timestamp(base + 1, 1, 15),
                      pd.Timestamp(base + 1, 2, 15)]
        try:
            oni = ssta_df.loc[djf_months, "oni"].mean()
        except KeyError:
            oni = np.nan
        phase = ("El Niño" if oni >= 0.5 else "La Niña" if oni <= -0.5 else "Neutral")
        rows.append({"year": yr, "oni_djf": oni, "enso_phase": phase})
    return pd.DataFrame(rows)


# 兩種指派各做一份合併表
merged_A = cholera.merge(annual_enso_phase(ssta, cholera["year"], False), on="year")
merged_B = cholera.merge(annual_enso_phase(ssta, cholera["year"], True), on="year")

order = ["El Niño", "Neutral", "La Niña"]
palette = {"El Niño": "#f8766d", "La Niña": "#619cff", "Neutral": "lightgray"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (mdf, title) in zip(
        axes, [(merged_A, "Definition A: DJF → that year"),
               (merged_B, "Definition B: same DJF → next year")]):
    sns.boxplot(data=mdf, x="enso_phase", y="detrended_cholera_cases", order=order,
                hue="enso_phase", palette=palette, legend=False,
                showfliers=False, width=0.5, ax=ax)
    sns.stripplot(data=mdf, x="enso_phase", y="detrended_cholera_cases", order=order,
                  color="black", size=4, jitter=0.12, ax=ax)
    means = mdf.groupby("enso_phase")["detrended_cholera_cases"].mean()
    ax.set_title(f"{title}\nEl Niño mean = {means.get('El Niño', float('nan')):.0f}")
    ax.set_xlabel(""); ax.axhline(0, color="gray", lw=0.6)
axes[0].set_ylabel("Detrended cholera cases")
plt.suptitle("Same data, two reasonable definitions → opposite conclusions", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# 把兩種定義下「聖嬰年平均霍亂」並排印出來，數字最有說服力
for label, mdf in [("A: DJF→that year", merged_A), ("B: DJF→next year", merged_B)]:
    means = mdf.groupby("enso_phase")["detrended_cholera_cases"].mean().round(0)
    print(label, "->", means.to_dict())
print("\n結論：只是換了 DJF 指派給哪一年，聖嬰年的霍亂訊號就從『明顯偏高』變成『幾乎沒差』。")
print("讀 paper 時務必確認：作者怎麼定義事件、怎麼對齊時間？相關 ≠ 因果。")

# C. 在地應用：龍洞浮標 + STL + 湧升指數
對應敘事文件: narrative/03_buoy_upwelling.md
時間預估：約 20 分鐘

資料是國海院 (NAMR) 提供的龍洞波浪浮標。這一段把前面學的工具用在在地資料上：
1. 浮標的季節循環**很強**（對比 B 段赤道的弱季節）。
2. 重用 **STL** 取季節/趨勢/殘差。
3. 由原始風資料算出物理量「**湧升指數 (Upwelling Index)**」。
4. 誠實地問：湧升和 SST 有沒有關係？(回扣故事①：別過度解讀、別挑窗)

註：圖上文字用英文 (避免 Colab 缺中文字型)；中文說明在 .md 與註解。

In [ ]:
# C0. 套件與工具
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

import ts_utils as ts

## C1. 浮標資料概覽：強烈的季節循環
龍洞浮標是**時頻 (hourly)** 資料、2010–2024，但有不少缺值。
先 resample 成月平均看大圖：SST 全年擺動約 10°C——**這裡季節才是主角**
（和 B 段赤道 Niño 3.4 只有 1.3°C 形成對比）。

In [ ]:
buoy = ts.load_buoy("Longdong")
monthly = buoy[["SST", "Wind", "Hs"]].resample("MS").mean()

fig, axes = plt.subplots(3, 1, figsize=(13, 6), sharex=True)
axes[0].plot(monthly.index, monthly["SST"], color="tab:red"); axes[0].set_ylabel("SST (°C)")
axes[1].plot(monthly.index, monthly["Wind"], color="tab:green"); axes[1].set_ylabel("Wind (m/s)")
axes[2].plot(monthly.index, monthly["Hs"], color="tab:blue"); axes[2].set_ylabel("Hs (m)")
axes[0].set_title("Longdong buoy — monthly mean (2010–2024)")
plt.tight_layout()
plt.show()

## C2. 重用 STL：強季節 + 殘差
把日平均 SST 丟給 STL（`period=365`）。和 B 段不同，這裡 seasonal 振幅很大。

⚠️ 看 trend 面板：它**不是**單調暖化、而是上下起伏。因為這裡**沒有鎖 `trend=`**（對比 A5），
STL 的趨勢較「軟」，把「年代際的緩慢振盪」也算進了趨勢——這**不代表**龍洞海溫真有那麼大的
年代際變化，而是 STL 固定週期分解的侷限（正是附錄 EEMD 想處理的問題）。
residual 面板裡的一些大塊則多半是**資料缺口**造成的插值假影。

正因如此，待會做湧升比較時，SST 端我們**不用這條 STL 殘差、改用「距平」**
（完整理由見 C4 與附錄 `appendix_ui_xcorr`）。這裡先示範 STL 在強季節資料上一樣拆得動。

In [ ]:
sst_d = (buoy["SST"].interpolate(limit=6)              # 補小缺口 (≤6 小時)
         .rolling(48, center=True, min_periods=24).mean()  # 48 小時平滑
         .resample("D").mean().dropna())               # 日平均

stl_sst = STL(sst_d, period=365, robust=True).fit()
fig = stl_sst.plot()
fig.set_size_inches(12, 8)
fig.suptitle("STL decomposition of Longdong daily SST", y=1.00)
plt.tight_layout()
plt.show()

## C3. 由原始風資料算「湧升指數 (Upwelling Index, UI)」
這是把**原始觀測**變成**物理量**的好例子：
風速 + 風向 → 風應力沿岸分量 → 除以 (海水密度 × 科氏參數) → 離岸 Ekman 輸送。
**UI 為正 = 有利湧升**（把底層冷水帶上來）。公式見 Huang et al. 2021。

In [ ]:
wind = buoy[["Wind", "Wind_Dir"]].interpolate(limit=6)   # 只內插數值欄
ui_h = ts.upwelling_index(wind, "Longdong", coast_angle=18.0)
ui_d = (ui_h.rolling(48, center=True, min_periods=24).mean()
        .resample("D").mean().dropna())

# UI 的季節氣候平均：夏季 (6–8 月) 偏正 = 有利湧升的風場
ui_clim = ui_d.groupby(ui_d.index.month).mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(ui_d.index, ui_d, color="tab:purple", lw=0.6)
axes[0].axhline(0, color="gray", lw=0.6)
axes[0].set_title("Daily upwelling index (UI)"); axes[0].set_ylabel("UI (m²/s)")
axes[1].bar(range(1, 13), ui_clim.values,
            color=["tab:red" if v > 0 else "tab:blue" for v in ui_clim.values])
axes[1].axhline(0, color="gray", lw=0.6)
axes[1].set_title("UI seasonal climatology (red>0 = upwelling-favorable)")
axes[1].set_xlabel("Month"); axes[1].set_ylabel("UI (m²/s)")
plt.tight_layout()
plt.show()

## C4. 沿用論文方法、用浮標驗證 (Huang et al. 2021)
論文用 CFSv2 風 + Himawari-8 衛星 SST，主結果是「湧升風天數↔衛星湧升天數 r=0.96」，
並指出「湧升訊號比風事件落後幾天」。這裡**沿用論文的 UI 公式 (β=18° 北段)**，
但改用國海院浮標資料驗證那句「湧升落後風場」。
機制：一陣**有利湧升的風** → 把底層冷水帶上來 → SST 過 1～2 天**下降** (預期負相關、UI 領先)。

做法（回扣主軸「比較異常」）：
- **UI** 取 STL **殘差**（去季節去趨勢）。
- **SST** 取**距平**（減掉浮標自身的月氣候平均）。
- 算兩者的**落後互相關**，聚焦 2016 年上升流季 (4–10 月)，與論文一致。

In [ ]:
# 先算好兩條序列
sst_clim = sst_d.groupby(sst_d.index.month).mean()        # 浮標自身的月氣候平均
sst_anom = sst_d - sst_d.index.month.map(sst_clim)        # SST 距平
ui_resid = STL(ui_d, period=365, robust=True).fit().resid  # UI 去季節去趨勢
window = slice("2016-04-10", "2016-10-15")                 # 聚焦上升流季 (與論文一致)

### C4a. 先用眼睛確認：UI 與 SST 距平是否反相位？
相關統計圖不直覺，所以**先疊合時間序列**：看 UI 衝高時，SST 距平是不是隨後往下掉。
這一步是在確認「換成浮標資料後，論文 Fig. 3 的 pattern 是否仍存在」——
確認看得到，才值得做後面的量化。

註：這張**先用原始 UI** 看趨勢比較直覺；下一步 C4b 量化時 UI 會改用「STL 殘差」
（去掉季節後雜訊較少），最佳落後相同、結論一致。

In [ ]:
ui_w = ui_d.loc[window]
anom_w = sst_anom.loc[window]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(ui_w.index, ui_w, color="tab:blue", label="Upwelling index (UI)")
ax.axhline(0, color="gray", lw=0.5)
ax.set_ylabel("UI (m²/s)", color="tab:blue"); ax.tick_params(axis="y", labelcolor="tab:blue")
ax2 = ax.twinx()
ax2.plot(anom_w.index, anom_w, color="tab:orange", label="SST anomaly")
ax2.set_ylabel("SST anomaly (°C)", color="tab:orange"); ax2.tick_params(axis="y", labelcolor="tab:orange")
ax.set_title("2016 upwelling season: UI vs SST anomaly  (UI spikes → SST dips shortly after)")
fig.tight_layout()
plt.show()

### C4b. 再量化：落後互相關
眼睛看到的 pattern，用數字確認：UI 領先 SST 幾天、相關多強。
UI 取 STL 殘差、SST 取距平，算落後互相關（這就是最後的量化結果）。

In [ ]:
corr_df = pd.concat([ui_resid.rename("ui_resid"),
                     sst_anom.rename("sst_anom")], axis=1, join="inner").dropna()
win = corr_df.loc[window]
lags, rs, ps, best = ts.lagged_xcorr(win["ui_resid"], win["sst_anom"], max_lag=15)
r_best = rs[list(lags).index(best)]

plt.figure(figsize=(10, 3.5))
plt.stem(lags, rs, basefmt="k-")
plt.axhline(0, color="gray", lw=0.6)
plt.axvline(best, color="red", ls="--", lw=1,
            label=f"best lag = +{best} d,  r = {r_best:.2f}")
plt.xlabel("Lag (days):  UI leads SST  →")
plt.ylabel("Correlation r")
plt.title("UI vs SST-anomaly cross-correlation (2016 upwelling season)")
plt.legend()
plt.tight_layout()
plt.show()

**怎麼讀這張圖（給第一次看互相關的人）：**
- **x 軸 = 落後天數**。`lag = +k` 的意思是「把 SST 往後挪 k 天去對齊 UI」，
  也就是在問：**今天的 UI 和 k 天後的 SST 有沒有關係**（正 lag = UI 領先 SST）。
- **y 軸 = 相關係數 r**（−1～+1）。**負的**代表「UI 高 → SST 低」，正是湧升的**降溫**。
- 所以我們專心看**右半邊（正 lag）有沒有明顯的負值**：這裡 **lag=+1、r≈−0.33** 最強，
  就是「有利湧升的風，領先海溫下降約 1 天」。
- ⚠️ r≈−0.33 是**中等偏弱**（單站、訊號雜）；而且相鄰 lag 的相關**彼此不獨立**，
  所以別只盯著某個 p<0.05，要看整體形狀（右半邊一路是負的）才可靠。

In [ ]:
print(f"best lag = +{best} day,  r = {r_best:.2f}")
print()
print("結果：最強的是 lag = +1 天、r ≈ -0.33 (負相關)——")
print("即『有利湧升的風』領先 SST 下降約 1 天，與 Huang et al. 2021『湧升落後風場』一致。")
print()
print("教學重點：")
print("- 沿用論文的 UI 方法 (公式、β=18°)，用國海院浮標資料就驗證得到湧升的降溫落後效應。")
print("- 方法回扣主軸：UI 取『STL 殘差』、SST 取『距平』，都是把可預期的部分拿掉、只比較異常。")
print("- 誠實區分：論文原始用衛星空間資料、主結果 r=0.96；我們是單站的改編驗證。")
print("- 這是聚焦上升流季 (4–10 月) 的單站結果，單站浮標是很好的地面驗證 (ground truth)。")

### C4c. 回扣 B 段：換個時間窗，結論會不會變？
B 段教過「定義/選擇會影響結論」。這裡換幾個時間窗算同一件事，看 best lag 與 r 穩不穩。

In [ ]:
for label, w in [("2016 上升流季", slice("2016-04-10", "2016-10-15")),
                 ("2018 上升流季", slice("2018-04-10", "2018-10-15")),
                 ("全紀錄 2010–2024", slice(None))]:
    sub = corr_df.loc[w] if w.start is not None else corr_df
    lg, r, p, bl = ts.lagged_xcorr(sub["ui_resid"], sub["sst_anom"], max_lag=15)
    print(f"{label:14s}: best lag = {bl:+d} d,  r = {r[list(lg).index(bl)]:+.2f}")
print()
print("→ 換個窗，best lag 與 r 就會變 (回扣 B 段：選擇會影響結論)。")
print("  所以我們才聚焦在『有物理意義的上升流季』，並誠實說明這是單站、弱訊號的驗證。")